In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/FYP_IIT/FYP_IIT_cleaned.csv', encoding='latin1')
df.head()

,gender,marital_status,academic_failure,faculty_environment,home_environment,age_group,academic_year,degree_program,peer_competition,lecturer_strictness,...,relaxation_time,exam_difficulty,exam_time_pressure,exam_stress,academic_confidence,career_confidence,decision_making,module_failure,exam_concern,employment_concern
0,Female,Single,Never,Satisfied,Yes,22-23,Year 4,B.B.A. (Hons) in Business Economics,4.0,3.0,...,1.0,4.0,4.0,5.0,2.0,2.0,4.0,4.0,4.0,4.0
1,Female,Single,Never,Satisfied,Yes,22-23,Year 4,B.B.A. (Hons) in International Business,3.0,2.0,...,4.0,3.0,3.0,3.0,2.0,2.0,4.0,4.0,3.0,4.0
2,Female,Single,Never,Very satisfied,Yes,22-23,Year 4,B.B.A. (Hons) in Human Resource Management,2.0,2.0,...,2.0,3.0,3.0,4.0,2.0,3.0,2.0,2.0,1.0,3.0
3,Male,Single,Once or twice,Satisfied,Yes,22-23,Year 4,B.B.A. (Hons) in Business Economics,4.0,3.0,...,1.0,5.0,3.0,2.0,1.0,2.0,3.0,4.0,2.0,3.0
4,Male,Single,Never,Satisfied,Yes,22-23,Year 4,B.B.A. (Hons) in Business Economics,5.0,4.0,...,2.0,4.0,4.0,4.0,1.0,2.0,1.0,4.0,4.0,5.0


In [ ]:
# Dimension 1: Academic Expectations
academic_expectations = [
    "peer_competition",
    "lecturer_strictness",
    "lecturer_expectations",
    "parental_expectations"
]
# Dimension 2: Workload & Examinations
workload_examinations = [
    "time_management",
    "curriculum_load",
    "assignment_load",
    "difficulty_catching_up",
    "relaxation_time",
    "exam_difficulty",
    "exam_time_pressure",
    "exam_stress"
]
# Dimension 3: Academic Self-Perception
academic_self_perception = [
    "academic_confidence",
    "career_confidence",
    "decision_making",
    "module_failure",
    "exam_concern",
    "employment_concern"
]

In [ ]:
dimensions = {
    "Academic Expectations": academic_expectations,
    "Workload & Examinations": workload_examinations,
    "Academic Self-Perception": academic_self_perception
}

for name, cols in dimensions.items():
    print(f"\n{name}:")
    for c in cols:
        print(" -", c)


Academic Expectations:
 - peer_competition
 - lecturer_strictness
 - lecturer_expectations
 - parental_expectations

Workload & Examinations:
 - time_management
 - curriculum_load
 - assignment_load
 - difficulty_catching_up
 - relaxation_time
 - exam_difficulty
 - exam_time_pressure
 - exam_stress

Academic Self-Perception:
 - academic_confidence
 - career_confidence
 - decision_making
 - module_failure
 - exam_concern
 - employment_concern


In [ ]:
# Combine all stress variables
all_items = academic_expectations + workload_examinations + academic_self_perception

In [ ]:
# Row-wise mean (one value per student)
df["overall_stress"] = df[all_items].mean(axis=1)

# Split overall stress by gender
male = df[df["gender"] == "Male"]["overall_stress"]
female = df[df["gender"] == "Female"]["overall_stress"]

In [ ]:
# Save updated dataset to CSV
df.to_csv("UoC_FMF_Inferential.csv", index=False)

# Download the file
from google.colab import files
files.download("UoC_FMF_Inferential.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from scipy.stats import ttest_ind
t_stat, p_value = ttest_ind(male, female, equal_var=False)

In [ ]:
print("t-statistic:", round(t_stat, 3))
print("p-value:", format(p_value, ".10f"))

t-statistic: 5.919
p-value: 0.0000000039


**ANOVA**

In [ ]:
from scipy.stats import f_oneway

# Create groups
group1 = df[df["academic_year"] == "Year 1"]["overall_stress"]
group2 = df[df["academic_year"] == "Year 2"]["overall_stress"]
group3 = df[df["academic_year"] == "Year 3"]["overall_stress"]
group4 = df[df["academic_year"] == "Year 4"]["overall_stress"]

# Perform ANOVA
f_stat, p_value = f_oneway(group1, group2, group3, group4)

print("F-statistic:", round(f_stat, 3))
print("p-value:", format(p_value, ".59f"))

F-statistic: 93.728
p-value: 0.00000000000000000000000000000000000000000000000000000002838


In [ ]:
# Step 1: Filter only Year 3 and Year 4
df_deg = df[df["academic_year"].isin(["Year 3", "Year 4"])]

# Step 2: Group by degree programme
groups = [group["overall_stress"].values
          for name, group in df_deg.groupby("degree_program")]

# Step 3: Perform ANOVA
from scipy.stats import f_oneway
f_stat_deg, p_value_deg = f_oneway(*groups)

print("F-statistic (Degree):", round(f_stat_deg, 3))
print("p-value:", format(p_value_deg, ".15f"))

F-statistic (Degree): 11.712
p-value: 0.000000000000026


**Tukeys Post Hoc**

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Tukey test for Academic Year
tukey = pairwise_tukeyhsd(
    endog=df["overall_stress"],
    groups=df["academic_year"],
    alpha=0.05
)

print(tukey)

Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj   lower  upper  reject
---------------------------------------------------
Year 1 Year 2   0.0517 0.3455 -0.0284 0.1319  False
Year 1 Year 3   0.3149    0.0  0.2402 0.3897   True
Year 1 Year 4   0.4223    0.0   0.348 0.4966   True
Year 2 Year 3   0.2632    0.0  0.1808 0.3456   True
Year 2 Year 4   0.3706    0.0  0.2886 0.4525   True
Year 3 Year 4   0.1074 0.0019  0.0307 0.1841   True
---------------------------------------------------


In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Run Tukey HSD
tukey_deg = pairwise_tukeyhsd(
    endog=df_deg["overall_stress"],      # dependent variable
    groups=df_deg["degree_program"],     # grouping variable
    alpha=0.05
)

print(tukey_deg)

                                             Multiple Comparison of Means - Tukey HSD, FWER=0.05                                              
                       group1                                              group2                       meandiff p-adj   lower   upper  reject
----------------------------------------------------------------------------------------------------------------------------------------------
                        B.B.A. (Hons) in Accounting            B.B.A. (Hons) in Business Administration  -0.3487    0.0 -0.5581 -0.1393   True
                        B.B.A. (Hons) in Accounting                 B.B.A. (Hons) in Business Economics   -0.197 0.0155 -0.3725 -0.0216   True
                        B.B.A. (Hons) in Accounting                            B.B.A. (Hons) in Finance   0.0917 0.6964 -0.0735  0.2568  False
                        B.B.A. (Hons) in Accounting B.B.A. (Hons) in Hospitality and Leisure Management  -0.2588 0.0049 -0.4691 -0.0484   True